# Βήμα 2: Καθαρισμός listings και σύνδεση με το sentiment

**Πριν ξεκινήσεις:** ανέβασε στον φάκελο `thesis` του Drive τα τρία `listings.csv.gz` (τα αναλυτικά, *Detailed Listings data*) με ονόματα:
`crete_listings.csv.gz`, `thessaloniki_listings.csv.gz`, `athens_listings.csv.gz`
(αν αποσυμπιεστούν σε `.csv`, δεν πειράζει, διαβάζονται κι έτσι).

Εδώ **δεν χρειάζεται GPU**: Χρόνος εκτέλεσης → Αλλαγή τύπου → **CPU**.

Αποτέλεσμα: ένας καθαρός πίνακας (μία γραμμή ανά κατάλυμα) στο `processed/listings_model.csv.gz`, έτοιμος για EDA και μοντέλα τιμής.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/thesis

import os, json
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 60)

REGIONS = ['crete', 'thessaloniki', 'athens']

## 1. Φόρτωση listings

In [ ]:
def find_file(region):
    for ext in ('.csv.gz', '.csv'):
        p = f'{region}_listings{ext}'
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Δεν βρέθηκε αρχείο listings για {region}')

frames = []
for r in REGIONS:
    df = pd.read_csv(find_file(r), low_memory=False)
    df['region'] = r
    frames.append(df)
    print(f'{r}: {len(df):,} καταλύματα, {df.shape[1]} στήλες')

raw = pd.concat(frames, ignore_index=True)

## 2. Καθαρισμός τιμής
Στα πρόσφατα snapshots του Inside Airbnb η τιμή λείπει για αρκετά καταλύματα (π.χ. όσα δεν έχουν διαθεσιμότητα). Το ποσοστό αυτό θα το αναφέρεις στη μεθοδολογία.

In [ ]:
L = raw.copy()
L['price'] = pd.to_numeric(L['price'].astype(str).str.replace(r'[$€,]', '', regex=True), errors='coerce')

print('Ποσοστό καταλυμάτων χωρίς τιμή ανά περιοχή:')
print(L.groupby('region')['price'].apply(lambda s: s.isna().mean()).round(3))

L = L[L['price'].notna() & (L['price'] > 0)]

# Αφαίρεση ακραίων τιμών: κρατάμε το 1ο-99ο εκατοστημόριο ανά περιοχή
q = L.groupby('region')['price'].quantile([0.01, 0.99]).unstack()
lo, hi = L['region'].map(q[0.01]), L['region'].map(q[0.99])
L = L[(L['price'] >= lo) & (L['price'] <= hi)].copy()
L['log_price'] = np.log(L['price'])
print(f'\nΚαταλύματα μετά τον καθαρισμό τιμής: {len(L):,}')

## 3. Χαρακτηριστικά καταλύματος, παροχές, οικοδεσπότης

In [ ]:
# Μπάνια από το κείμενο (π.χ. '1.5 shared baths', 'Half-bath')
bt = L['bathrooms_text'].fillna('').str.lower()
L['bathrooms_n'] = pd.to_numeric(bt.str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
L.loc[bt.str.contains('half') & L['bathrooms_n'].isna(), 'bathrooms_n'] = 0.5
L['bathroom_shared'] = bt.str.contains('shared').astype(int)

# Παροχές (amenities)
def parse_amenities(s):
    try:
        return [a.lower() for a in json.loads(s)]
    except Exception:
        return []

am = L['amenities'].apply(parse_amenities)
L['n_amenities'] = am.str.len()
KEY_AMENITIES = {
    'has_pool': 'pool', 'has_ac': 'air conditioning', 'has_free_parking': 'free parking',
    'has_sea_view': 'sea view', 'has_workspace': 'workspace', 'has_elevator': 'elevator',
    'has_balcony': 'balcony', 'has_self_checkin': 'self check-in',
}
for col, kw in KEY_AMENITIES.items():
    L[col] = am.apply(lambda lst: int(any(kw in a for a in lst)))

# Οικοδεσπότης και κράτηση
tf = {'t': 1, 'f': 0}
L['host_is_superhost'] = L['host_is_superhost'].map(tf)
L['instant_bookable'] = L['instant_bookable'].map(tf)
L['host_response_rate'] = pd.to_numeric(L['host_response_rate'].astype(str).str.rstrip('%'), errors='coerce') / 100
L['host_tenure_years'] = (pd.to_datetime(L['last_scraped']) - pd.to_datetime(L['host_since'])).dt.days / 365.25

## 4. Sentiment ανά κατάλυμα
Μέσος όρος score, ποσοστό αρνητικών κριτικών και πλήθος κριτικών που αναλύθηκαν (από το 2023 και μετά).

In [ ]:
sent = pd.concat([pd.read_csv(f'results/{r}_sentiment.csv.gz') for r in REGIONS], ignore_index=True)
sent['is_negative'] = (sent['sentiment_label'] == 'negative').astype(int)

sent_agg = (sent.groupby('listing_id')
            .agg(sent_mean=('sentiment_score', 'mean'),
                 sent_neg_share=('is_negative', 'mean'),
                 n_reviews_sent=('sentiment_score', 'size'))
            .reset_index())

L = L.merge(sent_agg, left_on='id', right_on='listing_id', how='left').drop(columns='listing_id')
L['n_reviews_sent'] = L['n_reviews_sent'].fillna(0).astype(int)
print('Καταλύματα με τουλάχιστον μία κριτική με sentiment:')
print(L.groupby('region')['n_reviews_sent'].apply(lambda s: (s > 0).mean()).round(3))

## 5. Επιλογή στηλών και αποθήκευση

In [ ]:
COLS = ['id', 'region', 'neighbourhood_cleansed', 'latitude', 'longitude',
        'room_type', 'property_type', 'accommodates', 'bedrooms', 'beds',
        'bathrooms_n', 'bathroom_shared', 'n_amenities', *KEY_AMENITIES,
        'host_is_superhost', 'host_response_rate', 'host_tenure_years',
        'calculated_host_listings_count', 'instant_bookable',
        'minimum_nights', 'availability_365', 'number_of_reviews',
        'review_scores_rating', 'review_scores_cleanliness',
        'review_scores_location', 'review_scores_value',
        'sent_mean', 'sent_neg_share', 'n_reviews_sent',
        'price', 'log_price']

missing = [c for c in COLS if c not in L.columns]
if missing:
    print('Στήλες που λείπουν από αυτό το snapshot (παραλείπονται):', missing)
model_df = L[[c for c in COLS if c in L.columns]]

os.makedirs('processed', exist_ok=True)
model_df.to_csv('processed/listings_model.csv.gz', index=False, compression='gzip')
print(f'Αποθηκεύτηκε: processed/listings_model.csv.gz {model_df.shape}')

## 6. Πρώτη ματιά

In [ ]:
aggs = {'listings': ('id', 'size'), 'median_price': ('price', 'median'),
        'mean_rating': ('review_scores_rating', 'mean'), 'mean_sentiment': ('sent_mean', 'mean'),
        'neg_share': ('sent_neg_share', 'mean'), 'superhost_share': ('host_is_superhost', 'mean')}
aggs = {k: v for k, v in aggs.items() if v[0] in model_df.columns}
summary = model_df.groupby('region').agg(**aggs).round(3)
summary

In [ ]:
# Ελλείποντα στοιχεία ανά στήλη (για το κεφάλαιο Δεδομένα)
model_df.isna().mean().sort_values(ascending=False).round(3).head(15)